# 臺大288校補查品質

參考文字、明列當期的接待校事實、臺大正式甄選與實際交換校區分開。4032列有狀態不代表4032列均核實。執行前先重建候選資料與audit輸出。

In [1]:
from pathlib import Path
import json
from collections import Counter
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sources/ntu/completion-2026-10-11.json").exists())
snapshot = json.loads((root / "sources/ntu/completion-2026-10-11.json").read_text())
review = json.loads((root / "docs/ntu-completion-20261011/stage2/all-school-review.json").read_text())
assert len(snapshot["schools"]) == 288
assert len(review) == 288 * 14
assert len({(r["portal_id"], r["field"]) for r in review}) == len(review)
print("School/field grain:", len(review))

School/field grain: 4032


In [2]:
before = json.loads((root / "docs/ntu-completion-20261011/before/summary.json").read_text())
after = json.loads((root / "docs/ntu-completion-20261011/stage2/review-summary.json").read_text())
for field in before["fields"]:
    print(field, before["fields"][field]["public_evidence_gap"], "->", after["fields"][field]["public_evidence_gap"])
assert after["fully_verified_target_cycle_schools"] == 0
print(Counter(r["issue_code"] for r in review))

home_selection 0 -> 0
eligibility 168 -> 162
language 180 -> 177
quota 282 -> 282
deadlines 194 -> 188
documents 186 -> 175
calendar 174 -> 170
courses 157 -> 146
housing 162 -> 70
cost 171 -> 150
insurance_visa 167 -> 150
life_sources 164 -> 46
coordinates 19 -> 3
photos 22 -> 3
Counter({'prior_reference_target_year_or_subfields_unconfirmed': 1170, 'asset_or_home_fact_checked_exchange_assignment_still_separate': 864, 'technical_transport_failure_prevents_check': 745, 'access_limited_no_verified_field_text': 465, 'official_list_blank_as_of_check': 282, 'reviewed_fact_in_original_scope': 274, 'discovery_lead_requires_semantic_review': 129, 'not_found_in_bounded_search_not_proof_unannounced': 87, 'version_conflict_requires_confirmation': 8, 'campus_identity_or_licensed_asset_unresolved': 6, 'officially_stated_future_update_pending': 1, 'public_reference_details_after_acceptance': 1})


In [3]:
observations = [r for school in snapshot["schools"] for r in school["observations"]]
assert all(r["selection_rule_confirmed"] is False for r in observations)
assert all(r["quote_source_id"] in snapshot["public_source_reviews"] for r in observations)
print("Manually reviewed observations:", len(observations))
print("Original applicability:", Counter(r["applicability_status"] for r in observations))
print("Conflict observations:", [(school["school_name"], r["field"]) for school in snapshot["schools"] for r in school["observations"] if r["applicability_status"] == "version_conflict"])

Manually reviewed observations: 293
Original applicability: Counter({'undated_public_reference': 194, 'target_year_explicit': 49, 'older_year_reference': 42, 'version_conflict': 7, 'explicitly_pending_announcement': 1})
Conflict observations: [('魁北克大學蒙特婁分校', 'documents'), ('魁北克大學蒙特婁分校', 'deadlines'), ('布達佩斯考文紐斯大學', 'housing'), ('優密歐大學', 'calendar'), ('聖安德魯斯大學', 'housing'), ('奧克蘭大學', 'housing'), ('奧克蘭大學', 'cost')]


原始正文保存在sources/ntu/reviewed-source-bodies-current-2026-10-11-part-01.zip等6份独立ZIP与current-manifest.json；217份hash核對正文對應191引用及校區／影像證據。第一批59原始正文另封存，舊ENS快取收據無hash的限制保留，09:59重新正常訪問另記新收據。技術錯誤、存取限制、接收後取得與明示待公告分開；4032列不代表全部已核實。